In [28]:
from dataset import load_snli_dataset, convert_to_dataframe
from tokenizer import train_tokenizer
from encoders import TokenIndicator, PairEncoder, BagofWords, MPNetEncoder
from sentence_transformers import SentenceTransformer
import numpy as np
import loglin
import cupy as cp


# 1 if compatible with GPU acceleration
print(cp.cuda.runtime.getDeviceCount())

1


In [29]:
dataset = load_snli_dataset()

In [30]:
train_df, validation_df, test_df = convert_to_dataframe(dataset)

In [31]:
corpus = list(set(train_df['premise'])) + list(set(train_df['hypothesis']))
#corpus

In [32]:
tokenizer = train_tokenizer(corpus, vocab_size=1000, min_frequency=2)

In [33]:
tokenizer.get_vocab_size()

1000

In [34]:
tokenizer.encode(corpus[0]).ids

[89, 121, 274, 311, 89, 676, 150, 387, 126, 104, 762, 62, 269]

In [35]:
## Token Indicator
encoder_tokenindicator = TokenIndicator(tokenizer)
encoding_tokenindicator = encoder_tokenindicator(corpus[0])
encoding_tokenindicator

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [36]:
## Bag of Words
encoder_bagofwords = BagofWords(tokenizer)
encoding_bagofwords = encoder_bagofwords(corpus[0])
encoding_bagofwords

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [37]:
row = train_df.iloc[0]
pair = ()

In [38]:
train_data = [((p, h), y) for p, h, y in zip(train_df["premise"], train_df["hypothesis"], train_df["label"])]
val_texts = list(zip(validation_df["premise"], validation_df["hypothesis"]))

In [39]:
## Sentence Transformer
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from pathlib import Path

sentence_model = SentenceTransformer(
    "sentence-transformers/nli-mpnet-base-v2",
    device="cuda",
)

# Encoder for the (premise, hiphotesis) pairs returned in batches
# Generated this alternative to PairEncoder because of efficiency
def encode_pairs(data, batch_size=256):
    # data contains (premise, hypothesis) pairs
    premises = [pair[0] for pair in data]
    hypotheses = [pair[1] for pair in data]

    # Encode all sentences in batches on GPU
    premise_vectors = sentence_model.encode(
        premises,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    hypothesis_vectors = sentence_model.encode(
        hypotheses,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True,
    )

    return np.concatenate(
        [premise_vectors, hypothesis_vectors],
        axis=1,
    )

def load_or_encode(filename, data):
    if Path(filename).exists():
        return np.load(filename)

    embeddings = encode_pairs(data)
    np.save(filename, embeddings)
    return embeddings

train_pairs = [text for text, label in train_data]
X_train = load_or_encode("X_train.npy", train_pairs)
print(X_train.shape)

y_train = np.array([label for text, label in train_data])

X_val = load_or_encode("X_val.npy", val_texts)
print(X_val.shape)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(549367, 1536)
(9842, 1536)


In [40]:
# Commented, this is the cell bellow but for CPU

#model = LogisticRegression(
#    solver="saga",
#    max_iter=100,
#)
#model.fit(X_train, y_train)

#preds = model.predict(X_val)

#print(accuracy_score(validation_df["label"], preds))

In [41]:
## SentenceTransformer + LogLin (This is for GPU, to use CPU use the implementation above)
from cuml.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Transfer embeddings to GPU
X_train_gpu = cp.asarray(X_train, dtype=cp.float32)
y_train_gpu = cp.asarray(y_train)

X_val_gpu = cp.asarray(X_val, dtype=cp.float32)

# Train logistic regression on GPU
model = LogisticRegression(
    max_iter=1000,
)

model.fit(X_train_gpu, y_train_gpu)

# Predict on GPU
preds_gpu = model.predict(X_val_gpu)

# Convert predictions back to NumPy
preds = cp.asnumpy(preds_gpu)

print(accuracy_score(validation_df["label"], preds))

0.6631782158097947


In [ ]:
## TokenIndicator + LogLin
pe = PairEncoder(TokenIndicator(tokenizer))
print("done pe")
model = loglin.fit(train_data, pe, max_iter=1000)
print("done model")
preds = loglin.predict(val_texts, model, pe)
print(accuracy_score(validation_df["label"], preds))

done pe


In [ ]:
pe = PairEncoder(TokenIndicator(tokenizer), overlap=True)
print("Done")
model = loglin.fit(train_data, pe, max_iter=1000)
preds = loglin.predict(val_texts, model, pe)
print(accuracy_score(validation_df["label"], preds))

In [ ]:
## BagofWords + LogLin
pe = PairEncoder(BagofWords(tokenizer))
model = loglin.fit(train_data, pe, max_iter=100)
preds = loglin.predict(val_texts, model, pe)
print(accuracy_score(validation_df["label"], preds))

In [ ]:
pe = PairEncoder(BagofWords(tokenizer), overlap=True)
model = loglin.fit(train_data, pe, max_iter=100)
preds = loglin.predict(val_texts, model, pe)
print(accuracy_score(validation_df["label"], preds))

In [ ]:
# Shared SentenceTransformer MLP setup (GPU only)
import importlib
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import models
importlib.reload(models)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable in this notebook kernel; check its PyTorch environment.")
st_device = torch.device("cuda")
print("Training on:", torch.cuda.get_device_name(st_device))
torch.manual_seed(42)

# Change limits to test smoke run
st_train_limit = None
st_val_limit = None
st_epochs = 3
st_batch_size = 256
st_rng = np.random.default_rng(42)
st_train_indices = st_rng.permutation(len(y_train))[:st_train_limit]
st_val_indices = st_rng.permutation(len(X_val))[:st_val_limit]
st_val_labels = validation_df["label"].to_numpy()
assert len(X_train) == len(y_train)
assert len(X_val) == len(st_val_labels)
assert X_train.shape[1] == X_val.shape[1]

# CPU tensors; only individual batches move to GPU.
st_train_set = TensorDataset(
    torch.as_tensor(np.asarray(X_train[st_train_indices]), dtype=torch.float32),
    torch.as_tensor(np.asarray(y_train)[st_train_indices], dtype=torch.long),
)
st_val_set = TensorDataset(
    torch.as_tensor(np.asarray(X_val[st_val_indices]), dtype=torch.float32),
    torch.as_tensor(st_val_labels[st_val_indices], dtype=torch.long),
)
st_train_loader = DataLoader(st_train_set, batch_size=st_batch_size, shuffle=True, num_workers=0)
st_val_loader = DataLoader(st_val_set, batch_size=st_batch_size, shuffle=False, num_workers=0)

# Both experiments reuse this loop, but create independent models.
def train_st_mlp(hidden_dims):
    torch.manual_seed(42)
    st_mlp = models.FeatureMLP(X_train.shape[1], hidden_dims=hidden_dims).to(st_device)
    st_optimizer = torch.optim.Adam(st_mlp.parameters(), lr=0.001)
    st_criterion = torch.nn.CrossEntropyLoss()

    for st_epoch in range(st_epochs):
        st_mlp.train()
        st_loss_sum = 0.0
        for st_features, st_labels in st_train_loader:
            st_features = st_features.to(st_device)
            st_labels = st_labels.to(st_device)
            st_optimizer.zero_grad(set_to_none=True)
            st_loss = st_criterion(st_mlp(st_features), st_labels)
            st_loss.backward()
            st_optimizer.step()
            st_loss_sum += st_loss.item() * len(st_labels)

        st_mlp.eval()
        st_predictions = []
        with torch.no_grad():
            for st_features, _ in st_val_loader:
                st_logits = st_mlp(st_features.to(st_device))
                st_predictions.append(st_logits.argmax(dim=1).cpu())
        st_predictions = torch.cat(st_predictions).numpy()
        st_accuracy = np.mean(st_predictions == st_val_labels[st_val_indices])
        print(f"Epoch {st_epoch + 1}: loss={st_loss_sum / len(st_train_set):.4f}, validation accuracy={st_accuracy:.4f}")
    return st_mlp, st_accuracy


In [ ]:
# SentenceTransformer + single-hidden-layer MLP (GPU only)
st_single_mlp, st_single_accuracy = train_st_mlp(hidden_dims=(128,))
st_single_accuracy

In [ ]:
# SentenceTransformer + multilayer MLP (GPU only)
st_multi_mlp, st_multi_accuracy = train_st_mlp(hidden_dims=(128, 64))
st_multi_accuracy